# Analyse exploratoire des ventes immobilières en France (DVF 2025)

**Objectif :** comprendre la structure et la qualité des données DVF, puis les facteurs qui influencent le prix d'un logement. Ce travail prépare un futur modèle de prédiction du prix d'un bien (régression).

**Source :** « Demandes de valeurs foncières géolocalisées » sur data.gouv.fr (DGFiP, enrichies par Etalab), fichier 2025, France entière.

> Ce premier notebook travaille sur un échantillon de 1 000 lignes pour découvrir les données avant d'extraire un département complet.

## 1. Chargement d'un échantillon

Le fichier complet compte plusieurs millions de lignes. On commence par les 1 000 premières (`nrows`) pour explorer rapidement.

Les codes département, commune et postal sont lus comme du **texte** (`dtype=str`) : ce sont des identifiants, pas des quantités. Lus comme des nombres, ils perdent leur zéro initial (`01158` devient `1158`) et la Corse (`2A`, `2B`) poserait problème.

In [55]:
import pandas as pd

pd.set_option("display.max_columns", None)

In [56]:
df = pd.read_csv(
    "../data/raw/dvf-2025.csv.gz",
    nrows=1000,
    dtype={"code_departement": str, "code_commune": str, "code_postal": str},
)

In [57]:
df.head()

,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,ancien_code_commune,ancien_nom_commune,id_parcelle,ancien_id_parcelle,numero_volume,lot1_numero,lot1_surface_carrez,lot2_numero,lot2_surface_carrez,lot3_numero,lot3_surface_carrez,lot4_numero,lot4_surface_carrez,lot5_numero,lot5_surface_carrez,nombre_lots,code_type_local,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
0,2025-1,2025-01-07,1,Vente,468000.0,NaN,NaN,FARGES,B078,01550,01158,Farges,01,NaN,NaN,011580000B0815,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,J,jardins,NaN,NaN,78.0,5.907192,46.170865
1,2025-1,2025-01-07,1,Vente,468000.0,454.0,NaN,RUE DE LA REPUBLIQUE,0090,01550,01158,Farges,01,NaN,NaN,011580000B0910,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3.0,Dépendance,0.0,0.0,S,sols,NaN,NaN,133.0,5.907199,46.170775
2,2025-1,2025-01-07,1,Vente,468000.0,454.0,NaN,RUE DE LA REPUBLIQUE,0090,01550,01158,Farges,01,NaN,NaN,011580000B0910,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1.0,Maison,111.0,5.0,S,sols,NaN,NaN,133.0,5.907199,46.170775
3,2025-2,2025-01-06,1,Vente,180000.0,NaN,NaN,LE VILLAGE,B158,01200,01257,Montanges,01,NaN,NaN,01257000AC0334,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,S,sols,NaN,NaN,46.0,5.801625,46.164569
4,2025-2,2025-01-06,1,Vente,180000.0,NaN,NaN,LE VILLAGE,B158,01200,01257,Montanges,01,NaN,NaN,01257000AC0338,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,J,jardins,NaN,NaN,17.0,5.801688,46.164608


In [58]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id_mutation                   1000 non-null   str    
 1   date_mutation                 1000 non-null   str    
 2   numero_disposition            1000 non-null   int64  
 3   nature_mutation               1000 non-null   str    
 4   valeur_fonciere               999 non-null    float64
 5   adresse_numero                455 non-null    float64
 6   adresse_suffixe               41 non-null     str    
 7   adresse_nom_voie              978 non-null    str    
 8   adresse_code_voie             978 non-null    str    
 9   code_postal                   978 non-null    str    
 10  code_commune                  1000 non-null   str    
 11  nom_commune                   1000 non-null   str    
 12  code_departement              1000 non-null   str    
 13  ancien_code_com

In [59]:
df[["code_departement", "code_commune", "code_postal"]].head()

,code_departement,code_commune,code_postal
0,01,01158,01550
1,01,01158,01550
2,01,01158,01550
3,01,01257,01200
4,01,01257,01200


**Observations :**

- 40 colonnes : identifiant de vente, date, nature de la vente, prix, adresse, codes géographiques, lots, type de local, surfaces, nature du terrain et coordonnées GPS.
- Les codes ont bien conservé leurs zéros (`01`, `01158`, `01550`).
- `code_postal` a 22 valeurs manquantes. Avant la correction, ces `NaN` forçaient pandas à convertir la colonne en décimal (`1550.0`), car une colonne d'entiers ne peut pas contenir de valeur manquante. En texte, le problème disparaît.

## 2. Des valeurs manquantes qui ont un sens

Beaucoup de colonnes ont des valeurs manquantes. Avant de les traiter, il faut comprendre **pourquoi** elles manquent.

In [60]:
df[df["adresse_numero"].isna()]["type_local"].value_counts(dropna=False)

type_local
NaN            508
Dépendance      23
Appartement     14
Name: count, dtype: int64

In [61]:
df[df["nature_culture"].isna()]["type_local"].value_counts(dropna=False)

type_local
Dépendance                                  124
Appartement                                  80
NaN                                          28
Maison                                        3
Local industriel. commercial ou assimilé      3
Name: count, dtype: int64

**Observations :**

- Sur les 545 lignes sans numéro d'adresse, 508 (93 %) n'ont pas de `type_local` : ce sont des **terrains** (champs, bois…), qui n'ont pas de numéro de rue.
- Les lignes sans `nature_culture` sont surtout des appartements et des dépendances : un appartement ne s'achète pas avec un terrain en propre.
- Les colonnes `ancien_code_commune` et `ancien_nom_commune` sont vides car elles ne sont remplies qu'en cas de changement de commune (fusion).

**Conclusion :** une ligne DVF décrit soit un **local bâti** (`type_local` rempli), soit un **terrain** (`nature_culture` rempli), parfois les deux. Ces valeurs manquantes sont structurelles : les remplacer par une moyenne inventerait des données.

*Question ouverte : 28 lignes n'ont ni `type_local` ni `nature_culture`.*

## 3. Une ligne n'est pas une vente

Les premières lignes partagent le même `id_mutation` et la même `valeur_fonciere`. Une vente peut porter sur plusieurs biens (appartement, cave, parking…) : DVF crée une ligne par bien et recopie le **prix total** de la vente sur chacune.

In [62]:
print(len(df)) 
print(df["id_mutation"].nunique())

1000
349


In [63]:
lignes_par_vente = df["id_mutation"].value_counts()
lignes_par_vente.head()

id_mutation
2025-92     23
2025-344    21
2025-62     18
2025-250    17
2025-282    15
Name: count, dtype: int64

In [64]:
print(f"ventes avec plus d’une ligne : {( lignes_par_vente > 1).sum()}")

ventes avec plus d’une ligne : 241


In [65]:
lignes_par_vente.value_counts().sort_index()

count
1     108
2     101
3      67
4      27
5      11
6      13
7       4
8       5
9       1
10      4
11      1
12      1
14      1
15      1
17      1
18      1
21      1
23      1
Name: count, dtype: int64

**Observations :**

- Les 1 000 lignes ne correspondent qu'à **349 ventes**, soit 2,9 lignes par vente en moyenne.
- **241 ventes sur 349 (69 %)** ont plus d'une ligne : la vente groupée est la norme.
- La distribution est **asymétrique à droite** : la plupart des ventes ont 1 à 3 lignes, quelques-unes jusqu'à 23.
- Les 22 ventes de 7 lignes ou plus ne représentent que 6 % des ventes, mais 25 % des lignes.

**Conséquence :** calculer des statistiques de prix ligne par ligne compterait chaque vente plusieurs fois, et un prix au m² calculé sur une seule ligne serait faux.

In [66]:
plus_grosse = lignes_par_vente.index[0]
df[df["id_mutation"] == plus_grosse]

,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,ancien_code_commune,ancien_nom_commune,id_parcelle,ancien_id_parcelle,numero_volume,lot1_numero,lot1_surface_carrez,lot2_numero,lot2_surface_carrez,lot3_numero,lot3_surface_carrez,lot4_numero,lot4_surface_carrez,lot5_numero,lot5_surface_carrez,nombre_lots,code_type_local,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
253,2025-92,2025-01-03,1,Vente,78296.0,NaN,NaN,LES COMBES,B013,01630,01354,St-Genis-Pouilly,01,NaN,NaN,013540000A0306,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,T,terres,NaN,NaN,4620.0,6.036914,46.265514
254,2025-92,2025-01-03,1,Vente,78296.0,NaN,NaN,LE MACLONAY,B034,01630,01354,St-Genis-Pouilly,01,NaN,NaN,013540000A0559,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,P,prés,NaN,NaN,4397.0,6.049646,46.267292
255,2025-92,2025-01-03,1,Vente,78296.0,NaN,NaN,LE MACLONAY,B034,01630,01354,St-Genis-Pouilly,01,NaN,NaN,013540000A0573,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,T,terres,NaN,NaN,3896.0,6.048718,46.270646
256,2025-92,2025-01-03,1,Vente,78296.0,NaN,NaN,LE MACLONAY,B034,01630,01354,St-Genis-Pouilly,01,NaN,NaN,013540000A0579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,BS,taillis sous futaie,NaN,NaN,2475.0,6.050179,46.271019
257,2025-92,2025-01-03,1,Vente,78296.0,NaN,NaN,LES SARCELLES,B054,01630,01354,St-Genis-Pouilly,01,NaN,NaN,013540000A0701,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,T,terres,NaN,NaN,6845.0,6.053063,46.265622
258,2025-92,2025-01-03,1,Vente,78296.0,NaN,NaN,LES GRILLETS,B026,01630,01354,St-Genis-Pouilly,01,NaN,NaN,013540000A1016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,T,terres,NaN,NaN,17330.0,6.035245,46.262504
259,2025-92,2025-01-03,1,Vente,78296.0,NaN,NaN,LES CHEMINETS,B010,01630,01354,St-Genis-Pouilly,01,NaN,NaN,013540000A1270,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,P,prés,NaN,NaN,465.0,6.033846,46.262976
260,2025-92,2025-01-03,1,Vente,78296.0,NaN,NaN,LA PRALAY,B044,01630,01354,St-Genis-Pouilly,01,NaN,NaN,01354000AD0049,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,P,prés,NaN,NaN,14710.0,6.043553,46.256935
261,2025-92,2025-01-03,1,Vente,78296.0,NaN,NaN,LA PRALAY,B044,01630,01354,St-Genis-Pouilly,01,NaN,NaN,01354000AD0160,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,P,prés,NaN,NaN,21.0,6.043500,46.260081
262,2025-92,2025-01-03,1,Vente,78296.0,NaN,NaN,LA PRALAY,B044,01630,01354,St-Genis-Pouilly,01,NaN,NaN,01354000AD0162,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,P,prés,NaN,NaN,12860.0,6.045189,46.258777


La plus grosse vente (23 lignes) n'a aucun `type_local` : c'est un ensemble de **terrains ruraux** (terres, prés, taillis sous futaie), pas un logement.

In [67]:
ventes_2_lignes = lignes_par_vente[lignes_par_vente == 2].index
df[df["id_mutation"].isin(ventes_2_lignes)][["id_mutation", "type_local", "surface_reelle_bati", "valeur_fonciere"]].head(20)

,id_mutation,type_local,surface_reelle_bati,valeur_fonciere
24,2025-6,NaN,NaN,543510.0
25,2025-6,NaN,NaN,543510.0
26,2025-7,Appartement,49.0,131937.3
27,2025-7,Dépendance,0.0,131937.3
28,2025-8,Dépendance,0.0,173000.0
29,2025-8,Appartement,78.0,173000.0
37,2025-11,Dépendance,0.0,630000.0
38,2025-11,Maison,100.0,630000.0
45,2025-13,Maison,144.0,314000.0
46,2025-13,Dépendance,0.0,314000.0


Dans les ventes à 2 lignes, on trouve surtout **un logement + une dépendance** (cave, parking). La dépendance a une surface bâtie de 0, et le prix est le total des deux.

*Limite : le prix inclut la dépendance, mais la surface n'est que celle du logement. Le prix au m² sera légèrement surestimé pour ces ventes.*

## 4. Types de vente

In [68]:
df["nature_mutation"].value_counts()

nature_mutation
Vente                                 948
Vente en l'état futur d'achèvement     52
Name: count, dtype: int64

L'échantillon contient 948 lignes de **Vente** classique et 52 de **Vente en l'état futur d'achèvement** (VEFA, achat sur plan dans le neuf). Le fichier complet contient aussi des échanges, adjudications et expropriations, dont les prix ne reflètent pas le marché classique.

## 5. Décision : stratégie de filtrage

Pour une première version (baseline), l'analyse portera sur **les ventes d'un seul logement**. Règles retenues (à valider sur un département complet) :

1. Garder les ventes qui contiennent **exactement un logement** (Maison ou Appartement), avec ou sans dépendances.
2. Garder uniquement `nature_mutation` = **Vente**.
3. Retirer les ventes sans prix ou sans surface du logement, **après avoir mesuré** combien elles représentent.

Option écartée pour l'instant : regrouper les lignes par vente. Elle pourra faire l'objet d'un second modèle, comparé au premier sur un même jeu de test.

**Prochaine étape :** extraire un département complet du fichier (`chunksize`) et l'enregistrer en Parquet.